# LoRA SFT 实验（Colab + PEFT + TRL）

本 Notebook 演示：
- 使用 **PEFT** 配置 LoRA adapter
- 使用 **TRL SFTTrainer** 做监督微调（SFT）
- 后台线程**并行**采样 GPU 显存占用与 GPU 利用率

> 建议在 Colab 中选择 **GPU 运行时**（T4 / L4 / A100）。

In [ ]:
# @title 1. 安装依赖
!pip install -q "transformers>=4.46.0" "datasets>=3.0.0" "accelerate>=1.0.0" \
    "peft>=0.13.0" "trl>=0.12.0" "bitsandbytes>=0.44.0" "nvidia-ml-py>=12.0.0" matplotlib pandas

In [ ]:
# @title 2. 导入 & GPU 环境检查
import gc
import threading
import time
from dataclasses import dataclass, field
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from trl import SFTConfig, SFTTrainer

try:
    import pynvml
    pynvml.nvmlInit()
    NVML_OK = True
except Exception as e:
    NVML_OK = False
    print(f"[warn] pynvml unavailable: {e}")

assert torch.cuda.is_available(), "请在 Colab 中选择 GPU 运行时后再运行本 Notebook。"
DEVICE_INDEX = 0
GPU_NAME = torch.cuda.get_device_name(DEVICE_INDEX)
print(f"GPU: {GPU_NAME}")
print(f"CUDA: {torch.version.cuda}")
print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
# @title 3. GPU Profiler（后台并行采样）

@dataclass
class GPUSample:
    timestamp: float
    step: int | None
    phase: str
    torch_allocated_gb: float
    torch_reserved_gb: float
    torch_max_allocated_gb: float
    gpu_mem_used_gb: float | None = None
    gpu_mem_total_gb: float | None = None
    gpu_util_pct: float | None = None
    gpu_mem_util_pct: float | None = None


class GPUProfiler:
    """在训练过程中用后台线程并行采样显存与 GPU 利用率。"""

    def __init__(self, device_index: int = 0, interval: float = 0.5):
        self.device_index = device_index
        self.interval = interval
        self.records: list[GPUSample] = []
        self.current_step: int | None = None
        self.current_phase: str = "idle"
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None
        self._handle = None
        if NVML_OK:
            self._handle = pynvml.nvmlDeviceGetHandleByIndex(device_index)

    def _sample_once(self) -> GPUSample:
        torch_allocated = torch.cuda.memory_allocated(self.device_index) / 1e9
        torch_reserved = torch.cuda.memory_reserved(self.device_index) / 1e9
        torch_max = torch.cuda.max_memory_allocated(self.device_index) / 1e9

        gpu_mem_used = gpu_mem_total = gpu_util = gpu_mem_util = None
        if self._handle is not None:
            mem = pynvml.nvmlDeviceGetMemoryInfo(self._handle)
            util = pynvml.nvmlDeviceGetUtilizationRates(self._handle)
            gpu_mem_used = mem.used / 1e9
            gpu_mem_total = mem.total / 1e9
            gpu_util = util.gpu
            gpu_mem_util = util.memory

        return GPUSample(
            timestamp=time.time(),
            step=self.current_step,
            phase=self.current_phase,
            torch_allocated_gb=torch_allocated,
            torch_reserved_gb=torch_reserved,
            torch_max_allocated_gb=torch_max,
            gpu_mem_used_gb=gpu_mem_used,
            gpu_mem_total_gb=gpu_mem_total,
            gpu_util_pct=gpu_util,
            gpu_mem_util_pct=gpu_mem_util,
        )

    def _loop(self):
        while not self._stop.is_set():
            self.records.append(self._sample_once())
            self._stop.wait(self.interval)

    def start(self, phase: str = "train"):
        self.current_phase = phase
        torch.cuda.reset_peak_memory_stats(self.device_index)
        self.records.clear()
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join()
            self._thread = None

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame([r.__dict__ for r in self.records])

    def print_summary(self, title: str = "Profile Summary"):
        df = self.to_dataframe()
        if df.empty:
            print(f"[{title}] no samples collected")
            return
        print(f"\n=== {title} ===")
        print(f"samples: {len(df)}")
        print(f"torch allocated  max/mean: {df['torch_allocated_gb'].max():.2f} / {df['torch_allocated_gb'].mean():.2f} GB")
        print(f"torch reserved   max/mean: {df['torch_reserved_gb'].max():.2f} / {df['torch_reserved_gb'].mean():.2f} GB")
        print(f"torch peak alloc max     : {df['torch_max_allocated_gb'].max():.2f} GB")
        if df['gpu_mem_used_gb'].notna().any():
            print(f"NVML VRAM used   max/mean: {df['gpu_mem_used_gb'].max():.2f} / {df['gpu_mem_used_gb'].mean():.2f} GB")
        if df['gpu_util_pct'].notna().any():
            print(f"GPU util %       max/mean: {df['gpu_util_pct'].max():.0f} / {df['gpu_util_pct'].mean():.0f} %")


class GPUProfileCallback(TrainerCallback):
    """将 Trainer 训练过程与 GPUProfiler 绑定。"""

    def __init__(self, profiler: GPUProfiler):
        self.profiler = profiler

    def on_train_begin(self, args, state, control, **kwargs):
        self.profiler.start(phase="train")

    def on_step_end(self, args, state, control, **kwargs):
        self.profiler.current_step = state.global_step

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not self.profiler.records:
            return
        last = self.profiler.records[-1]
        loss = logs.get("loss") if logs else None
        loss_str = f"loss={loss:.4f} | " if loss is not None else ""
        util_str = f"gpu_util={last.gpu_util_pct:.0f}% | " if last.gpu_util_pct is not None else ""
        mem_str = f"torch_alloc={last.torch_allocated_gb:.2f}GB"
        if last.gpu_mem_used_gb is not None:
            mem_str += f", nvml_used={last.gpu_mem_used_gb:.2f}GB"
        print(f"[step {state.global_step:4d}] {loss_str}{util_str}{mem_str}")

    def on_train_end(self, args, state, control, **kwargs):
        self.profiler.stop()


profiler = GPUProfiler(device_index=DEVICE_INDEX, interval=0.5)
print("GPUProfiler ready.")

In [ ]:
# @title 4. 实验配置
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # Colab T4 友好
OUTPUT_DIR = "./lora-qwen-output"
MAX_SAMPLES = 200          # 先用小样本快速跑通
MAX_LENGTH = 512

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

BATCH_SIZE = 2
GRAD_ACCUM = 4
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4       # LoRA 常用较高学习率
LOGGING_STEPS = 5

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    max_length=MAX_LENGTH,
    packing=False,
    gradient_checkpointing=True,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
)

print(peft_config)
print(training_args)

In [ ]:
# @title 5. 加载数据集
raw_dataset = load_dataset("trl-lib/Capybara", split=f"train[:{MAX_SAMPLES}]")
print(raw_dataset)
print(raw_dataset[0].keys())
print(raw_dataset[0])

In [ ]:
# @title 6. 加载模型 & 构建 SFTTrainer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
if training_args.gradient_checkpointing:
    model.enable_input_require_grads()

# 训练前 baseline profile
profiler.start(phase="baseline")
time.sleep(2)
profiler.stop()
profiler.print_summary("Baseline (model loaded, before train)")
baseline_df = profiler.to_dataframe()
profiler.records.clear()

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Before PEFT | trainable: {trainable_params:,} / total: {total_params:,}")

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=raw_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
    callbacks=[GPUProfileCallback(profiler)],
)

model.print_trainable_parameters()

In [ ]:
# @title 7. 开始 LoRA 微调
train_result = trainer.train()
print(train_result)
profiler.print_summary("Training")

In [ ]:
# @title 8. 可视化 Profile 曲线
from IPython.display import display

profile_df = profiler.to_dataframe()
if profile_df.empty:
    print("No profile data.")
else:
    profile_df["elapsed_s"] = profile_df["timestamp"] - profile_df["timestamp"].iloc[0]
    display(profile_df.tail(10))

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    axes[0].plot(profile_df["elapsed_s"], profile_df["torch_allocated_gb"], label="torch allocated (GB)")
    axes[0].plot(profile_df["elapsed_s"], profile_df["torch_reserved_gb"], label="torch reserved (GB)", alpha=0.8)
    if profile_df["gpu_mem_used_gb"].notna().any():
        axes[0].plot(profile_df["elapsed_s"], profile_df["gpu_mem_used_gb"], label="NVML used (GB)", linestyle="--")
    axes[0].set_ylabel("Memory (GB)")
    axes[0].set_title("VRAM during LoRA fine-tuning")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    if profile_df["gpu_util_pct"].notna().any():
        axes[1].plot(profile_df["elapsed_s"], profile_df["gpu_util_pct"], color="tab:orange", label="GPU util (%)")
        axes[1].set_ylim(0, 100)
    else:
        axes[1].text(0.5, 0.5, "NVML unavailable", ha="center", va="center", transform=axes[1].transAxes)
    axes[1].set_xlabel("Elapsed time (s)")
    axes[1].set_ylabel("GPU Util (%)")
    axes[1].set_title("GPU utilization during LoRA fine-tuning")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# @title 9. 保存 LoRA adapter & 简单推理测试
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

prompt = "用一句话解释 LoRA 是什么？"
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

model.eval()
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False)

print("\n=== Inference ===")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 下一步可尝试

1. 调大 `MAX_SAMPLES` / `NUM_EPOCHS` 观察显存变化
2. 对比不同 `LORA_R`（如 4 / 16 / 32）的 trainable params 与峰值显存
3. 开启 4bit QLoRA（`BitsAndBytesConfig`）进一步降低显存
4. 将 `profiler.print_summary()` 输出记录到表格，做多组实验对比